# `stat_testing` partial reconstruction

This notebook combines:
- the only complete Cursor history snapshot recovered from March 24, 2026
- later code fragments recovered from March 25-26 Jupyter tracebacks

The later cells are a best-effort reconstruction and may need adjustment.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import gsw


In [ ]:
path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch = xr.open_dataset(path, consolidated=True)


In [ ]:
surface = llc_patch.isel(k=0)
surface_theta = surface.Theta
surface_theta_vals = surface_theta.isel(time=slice(9216, 9260)).stack(vals=surface_theta.dims)


In [ ]:
plt.hist(surface_theta_vals, bins=500, density=True)
plt.show()


## Recovered later work

The next cells are reconstructed from Jupyter log snippets. They are useful as a starting point, but they were not recovered from a clean notebook save.


In [ ]:
# Depth-colored Theta PDF work recovered from traceback snippets.
# `Theta` may need to be defined or reshaped the way you had it originally.
cm = [plt.cm.viridis(i / len(Theta)) for i in range(len(Theta))]
for k in range(len(Theta)):
    plt.hist(Theta[k], bins=500, density=True, color=cm[k], alpha=0.99)
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=0, vmax=len(Theta) - 1))
sm.set_array([])
plt.colorbar(sm, label='Depth')
plt.title('Theta PDF for all depths')
plt.show()


In [ ]:
# First GSW attempt from the logs.
pressure = gsw.p_from_z(-llc_patch.Z.values, llc_patch['YC'].values)
SA = gsw.SA_from_SP(llc_patch.Salt, pressure, llc_patch['XC'], llc_patch['YC'])
CT = gsw.CT_from_pt(SA, llc_patch.Theta)


In [ ]:
# Second GSW attempt from the logs with explicit broadcasting.
pressure = gsw.p_from_z(
    -llc_patch.Z.values[:, np.newaxis, np.newaxis],
    llc_patch['YC'].values[np.newaxis, :, :],
)
SA = gsw.SA_from_SP(
    llc_patch['Salt'],
    pressure,
    llc_patch['XC'].values[np.newaxis, np.newaxis, :],
    llc_patch['YC'].values[np.newaxis, :, np.newaxis],
)
CT = gsw.CT_from_pt(SA, llc_patch.Theta)
N2 = gsw.stability.Nsquared(SA, CT, pressure)


In [ ]:
# Another GSW attempt recovered from the logs.
pressure = gsw.p_from_z(-llc_patch.Z.values, llc_patch['YC'].isel(i=0).values)
SA = gsw.SA_from_SP(llc_patch.Salt, pressure, llc_patch['XC'], llc_patch['YC'])
CT = gsw.CT_from_pt(SA, llc_patch.Theta)


In [ ]:
# Broadcast-shape attempt recovered from traceback comments.
Z_grid = -llc_patch.Z.values[:, np.newaxis]          # expected shape: (51, 1)
lat_grid = llc_patch['YC'].values[np.newaxis, :]     # expected shape: (1, 720)
pressure = gsw.p_from_z(Z_grid, lat_grid)
pressure = pressure[np.newaxis, :, :, np.newaxis]


In [ ]:
# N2 histogram work recovered from tracebacks.
# The original loop used range(0, 51) and hit an index error; this version uses len(N2_hist).
cm = [plt.cm.viridis(i / len(N2_hist)) for i in range(len(N2_hist))]
for k in range(len(N2_hist)):
    plt.hist(N2_hist[k], bins=500, density=True, color=cm[k], alpha=0.2)
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=0, vmax=len(N2_hist) - 1))
sm.set_array([])
plt.colorbar(sm, label='Depth')
plt.title('N2 PDF for all depths')
plt.show()


In [ ]:
# S2 histogram work recovered from tracebacks.
cm = [plt.cm.viridis(i / len(S2_hist)) for i in range(len(S2_hist))]
for k in range(len(S2_hist)):
    plt.hist(S2_hist[k], bins=500, density=True, color=cm[k], alpha=0.2)
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=0, vmax=len(S2_hist) - 1))
sm.set_array([])
plt.colorbar(sm, label='Depth')
plt.title('S2 PDF for all depths')
plt.show()


In [ ]:
# Ri histogram work recovered from tracebacks.
cm = [plt.cm.viridis(i / len(Ri_hist)) for i in range(len(Ri_hist))]
for k in range(len(Ri_hist)):
    plt.hist(Ri_hist[k], bins=500, density=True, color=cm[k], alpha=0.2)
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=0, vmax=len(Ri_hist) - 1))
sm.set_array([])
plt.colorbar(sm, label='Depth')
plt.title('Ri PDF for all depths')
plt.show()
